Трейнинг + тест CNN с аугментацией fear и disgust

In [ ]:
!unzip archive.zip

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [ ]:
class load_dataset(Dataset):

    def __init__(self, root_dir):
        self.classes = sorted(os.listdir(root_dir))
        self.labels = []
        self.images = []
        for i, em in enumerate(self.classes):
            em_dir = os.path.join(root_dir, em)
            for img in os.listdir(em_dir):
                img_path = os.path.join(em_dir, img)
                image = Image.open(img_path)
                img_array = np.array(image)
                img_tensor = torch.from_numpy(img_array) / 255.0
                img_tensor = img_tensor.unsqueeze(0)
                self.images.append(img_tensor)
                self.labels.append(i)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

In [ ]:
print(sorted(os.listdir('test')))

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(64 * 6 * 6, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7),
        )

    def forward(self, x):
        return self.net(x)

Добавила агументацию fear и disgust на основе полученных метрик после теста ('flip', 'rotate', 'brightness', 'contrast')

In [ ]:
from torch.utils.data import random_split
import torchvision.transforms.functional as TF
import random

trainv_dataset = load_dataset("train")
test_dataset = load_dataset("test")

gen = torch.Generator().manual_seed(42)
tr_size = int(0.8 * len(trainv_dataset))
train_dataset, val_dataset = random_split(trainv_dataset, [tr_size, len(trainv_dataset) - tr_size ] , generator = gen)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class Augmentation(Dataset):
    def __init__(self, df, ixes):
        self.df = df
        self.ixes = set(ixes)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img, label = self.df[idx]
        if label in self.ixes:
            aug_type = random.choice(['flip', 'rotate', 'brightness', 'contrast'])
            if aug_type == 'flip':
                img = TF.hflip(img)
            elif aug_type == 'rotate':
                factor = random.uniform(-10, 10)
                img = TF.rotate(img, factor)
            elif aug_type == 'brightness':
                factor = random.uniform(0.8, 1.2)
                img = TF.adjust_brightness(img, factor)
            elif aug_type == 'contrast':
                factor = random.uniform(0.8, 1.2)
                img = TF.adjust_contrast(img, factor)
        return img, label

ixes = [1, 2]

train_dataset = Augmentation(train_dataset, ixes)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
best_acc = 0

os.makedirs("../best_models2", exist_ok=True)

In [ ]:
model = CNN()

batch_size=64
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
criterion = nn.CrossEntropyLoss()
patience, count = 2, 0

for i in range(11):
    model.train()
    for images, labels in train_loader:
        images, labels=images , labels
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels=images, labels
            output = model(images)
            _, pred = torch.max(output, 1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

    scheduler.step()
    res = 100*correct/total
    print(f"epoch {i +1}, acc: {res:.2f}%")

    if res > best_acc:
        best_acc = res
        torch.save(model.state_dict(), f"../best_models2/model_checkpoint_{res:.2f}.pth")
        count=0
    else:
      count+=1
      if count>=patience: break

epoch 1, acc: 44.51%
epoch 2, acc: 46.67%
epoch 3, acc: 53.08%
epoch 4, acc: 53.71%
epoch 5, acc: 53.59%
epoch 6, acc: 57.30%
epoch 7, acc: 57.37%
epoch 8, acc: 57.80%
epoch 9, acc: 57.96%
epoch 10, acc: 58.08%
epoch 11, acc: 58.46%


In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
checkpoint = torch.load("../best_models2/model_checkpoint_58.46.pth")

model = CNN()
model.load_state_dict(checkpoint)
model.eval()

transform = transforms.ToTensor()
classes = sorted(os.listdir("test"))
correct, total = 0, 0

y_true, y_pred = [], []

for i, claas in enumerate(classes):
    p = os.path.join("test", claas)
    for fil in os.listdir( p):
        img = Image.open(os.path.join(p, fil))
        img_tensor = transform(img).unsqueeze(0)
        with torch.no_grad():
            pred = model(img_tensor).argmax(dim=1).item()
        y_true.append(i)
        y_pred.append(pred)

print(f"Accuracy: {accuracy_score(y_true, y_pred)}%")

Accuracy: 0.5778768459180831%


In [ ]:
print(classes)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


Точнее 57.78 конечно))

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, target_names=classes)
print(report)

              precision    recall  f1-score   support

       angry       0.49      0.49      0.49       958
     disgust       0.69      0.20      0.31       111
        fear       0.44      0.17      0.25      1024
       happy       0.78      0.83      0.80      1774
     neutral       0.51      0.58      0.54      1233
         sad       0.42      0.53      0.47      1247
    surprise       0.68      0.77      0.72       831

    accuracy                           0.58      7178
   macro avg       0.57      0.51      0.51      7178
weighted avg       0.57      0.58      0.56      7178



Аугментация совсем не помогла :((